# 13 — Naive baseline

A zero-information reference for every model notebook (`05`–`10`). For each of
the 7 event fields, always predict the same constant, estimated from the
**training set only**:

- **mode** for the categorical fields (`is_home`, `event`, `result`) — scored by accuracy
- **median** for the numeric fields (`start_x`, `start_y`, `time_elapsed`, `duration`) — scored by MAE (the median is the MAE-optimal constant)

It also reports the **cross-entropy of each field's training marginal**, which
sums to a number directly comparable to the disjoint-heads model's summed
per-field cross-entropy loss.

Rows are grouped exactly like the models see them: the rows predicting field
`event_cols[s]` are the ones with `step == s`, and `target` holds the true
value (with the boundary sentinel `-2` already remapped to `106` by `03_splits`).

In [1]:
import json

import numpy as np
import pandas as pd

train = pd.read_csv("data/train_set.csv")
val = pd.read_csv("data/val_set.csv")
test = pd.read_csv("data/test_set.csv")

with open("data/event_masks.json") as f:
    masks = json.load(f)
with open("data/category_lookup.json") as f:
    lookup = {col: {int(k): v for k, v in m.items()} for col, m in json.load(f).items()}

event_cols = ["is_home", "event", "start_x", "start_y", "time_elapsed", "duration", "result"]
categorical = ["is_home", "event", "result"]

In [2]:
def field_targets(df, field):
    """True values of `field` across the split — the `target` column on the
    rows where that field is the one being predicted."""
    s = event_cols.index(field)
    return df.loc[df["step"] == s, "target"].to_numpy()

## Point predictions: mode (categorical) / median (numeric)

In [3]:
def mode_baseline(field):
    tr = field_targets(train, field)
    vals, counts = np.unique(tr, return_counts=True)
    mode = int(vals[counts.argmax()])

    row = {"field": field, "predict": f"{mode} = {lookup[field].get(mode, mode)}"}
    for name, df in [("val", val), ("test", test)]:
        y = field_targets(df, field)
        row[f"{name}_acc"] = (y == mode).mean()
    return row


def median_baseline(field):
    med = float(np.median(field_targets(train, field)))

    row = {"field": field, "predict": f"{med:g}"}
    for name, df in [("val", val), ("test", test)]:
        y = field_targets(df, field)
        row[f"{name}_mae"] = np.abs(y - med).mean()
    return row

In [4]:
point = pd.DataFrame(
    mode_baseline(f) if f in categorical else median_baseline(f)
    for f in event_cols
)
point

,field,predict,val_acc,test_acc,val_mae,test_mae
0,is_home,1 = home,0.517566,0.515632,NaN,NaN
1,event,1 = pass,0.430463,0.433047,NaN,NaN
2,start_x,54,NaN,NaN,20.627935,20.789790
3,start_y,34,NaN,NaN,17.596798,17.629967
4,time_elapsed,1,NaN,NaN,2.192153,2.232852
5,duration,1,NaN,NaN,1.054576,1.057195
6,result,0 = success,0.826649,0.823841,NaN,NaN


## Distributional: cross-entropy of the training marginal

For each field, the "prediction" is the full training-set class distribution
(Laplace-smoothed so classes unseen in train still get a finite log-prob).
The per-field cross-entropies sum to the quantity the disjoint-heads model
reports as its loss.

In [5]:
def marginal_ce(field):
    # event_masks lists a field's legal values in the pre-split space (boundary
    # as -2); target uses 106 for boundary, so align on that.
    classes = [106 if v == -2 else v for v in masks[field]]
    pos = {c: i for i, c in enumerate(classes)}

    counts = np.bincount([pos[v] for v in field_targets(train, field)], minlength=len(classes)).astype(float)
    p = (counts + 1) / (counts + 1).sum()

    row = {"field": field, "n_classes": len(classes)}
    for name, df in [("val", val), ("test", test)]:
        y = field_targets(df, field)
        row[f"{name}_ce"] = float(np.mean([-np.log(p[pos[v]]) for v in y]))
    return row

In [6]:
ce = pd.DataFrame(marginal_ce(f) for f in event_cols)
ce.loc["total", ["val_ce", "test_ce"]] = ce[["val_ce", "test_ce"]].sum()
ce

,field,n_classes,val_ce,test_ce
0,is_home,3.0,0.702846,0.703074
1,event,24.0,1.559665,1.556590
2,start_x,107.0,4.568005,4.568745
3,start_y,70.0,4.225873,4.225596
4,time_elapsed,106.0,1.881414,1.884877
5,duration,37.0,1.536113,1.535872
6,result,7.0,0.487701,0.493878
total,NaN,NaN,14.961616,14.968631


## For reference: naive baseline vs. disjoint-heads model (v5), val

v5 column from `evaluations/mlp_v5_disjoint.md` (kept in `scratch/`), measured
the same way — every `step == s` row, no clean-event filter. The v5 **test**
numbers come from re-running `10_mlp_v5_embeddings_disjoint_heads.ipynb` on the
test split.

| field | metric | naive (val) | v5 (val) |
|---|---|---|---|
| is_home | acc | 51.8% | 95.1% |
| event | acc | 43.0% | 75.5% |
| result | acc | 82.7% | 88.4% |
| start_x | MAE | 20.63 | 6.50 |
| start_y | MAE | 17.60 | 6.89 |
| time_elapsed | MAE | 2.19 | 0.91 |
| duration | MAE | 1.05 | 0.75 |
| **all 7** | summed CE | **14.96** | **9.573** |

`result` accuracy is the one field where the naive baseline is already
competitive — "always `success`" is right 82.7% of the time because the class
is that imbalanced (same observation as the v1 evaluation). Everywhere else the
model is well clear of the floor; the summed cross-entropy gap (~5.4 nats) is
the compact version of that.